# Seeded DLA VAE: training and evaluation

Train the query-conditioned VAE on DLA trees and evaluate it against the simulator and an
independent baseline. The query is `query(seed, t) -> (parent, direction)`: node `t`'s parent and
which side of the parent it sits on. The parent and direction heads are independent given the
latent and the query.

See `docs/dla.md` for the simulator and `docs/dla-plan.md` for the plan.

Section 1 trains and evaluates one profile in detail. Section 2 repeats the evaluation across
sizes and compares the results in one table.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from dla.config import PROFILES
from dla.evaluate import (
    collision_experiment,
    consistency_experiment,
    fractal_dimension_experiment,
    reconstruction_experiment,
    samples_experiment,
    timing_experiment,
    tree_statistics_experiment,
)
from dla.generate import generators_from_run
from dla.train import train

## 1. One profile

Profiles: `smoke` (16 nodes), `n64`, `n128`, `n256`. `smoke` only checks that everything runs.

In [ ]:
PROFILE = "smoke"
config = PROFILES[PROFILE]
run = train(config)
generators = generators_from_run(run)

### Reconstruction

Encode each test tree, decode from the mean latent, and score the true parents and directions.
This shows whether the latent carries tree-specific information. It is not a generation metric.
A direction cross-entropy at the uniform value (ln 4 ≈ 1.386) means the direction head learned
nothing.

In [ ]:
reconstruction = reconstruction_experiment(run)

### Samples

Simulator test trees against trees generated from fresh seeds. Red nodes land on a square that
an earlier node already occupies.

In [ ]:
collections = samples_experiment(run, generators["vae"], generators["independent"])

### Collisions

The simulator never collides. This is the most direct check of whether independent queries
agree on one geometry. The VAE should beat the independent baseline here if its latent
coordinates the queries.

In [ ]:
collisions = collision_experiment(collections)

### Fractal dimension

The radius of gyration of the first `k` nodes grows like `k^(1/D)`. DLA on a grid gives D of
about 1.65–1.72. Below a few thousand nodes the estimate is noisy. Trees with collisions are
measured as rebuilt, overlaps included, so heavy collisions push D up.

In [ ]:
dimension = fractal_dimension_experiment(collections, fit_min=max(4, config.num_nodes // 16))

### Tree shape

Depth, leaves, and branching from the parent arrays alone. On the grid, a node other than the
root has at most 3 free sides, so `max_children` above 3 is impossible in real DLA.

In [ ]:
tree_statistics = tree_statistics_experiment(collections)

### Consistency and speed

Consistency: answers must not depend on query order or on other seeds' queries.

Speed: one model query for node `t` against simulating a cluster up to node `t`. The model runs
one query at a time on CPU.

In [ ]:
consistency = consistency_experiment(generators["vae"])
timing = timing_experiment(generators["vae"], num_nodes=config.num_nodes, threads=config.threads)

## 2. Across sizes

Train and evaluate each profile in `SIZES`, then compare. This retrains every profile, so it is
off by default. Set `RUN_SWEEP = True` to run it.

In [ ]:
RUN_SWEEP = False
SIZES = ["n64", "n128", "n256"]


def summarize(profile):
    config = PROFILES[profile]
    run = train(config, report=None, show=False)
    generators = generators_from_run(run)
    collections = samples_experiment(run, generators["vae"], generators["independent"], show=False)
    reconstruction = reconstruction_experiment(run, show=False)
    collisions = collision_experiment(collections, show=False)
    dimension = fractal_dimension_experiment(
        collections, fit_min=max(4, config.num_nodes // 16), show=False
    )
    tree = tree_statistics_experiment(collections, show=False)
    timing = timing_experiment(
        generators["vae"], num_nodes=config.num_nodes, threads=config.threads, show=False
    )
    return {
        "nodes": config.num_nodes,
        "selected_epoch": run.selected_epoch,
        "train_seconds": run.elapsed_seconds,
        "direction_ce": reconstruction["mean_latent_test_direction_cross_entropy"],
        "parent_accuracy": reconstruction["mean_latent_test_parent_accuracy"],
        "collisions": {name: row["mean_collision_fraction"] for name, row in collisions.items()},
        "dimension": {name: row["fractal_dimension"] for name, row in dimension.items()},
        "max_depth": {name: row["max_depth"] for name, row in tree.items()},
        "query_ms": timing["per_arrival"][-1]["median_query_ms"],
        "simulate_ms": timing["per_arrival"][-1]["median_simulate_to_arrival_ms"],
    }


sweep = {profile: summarize(profile) for profile in SIZES} if RUN_SWEEP else {}

In [ ]:
if sweep:
    names = ["Simulator", "VAE", "Independent"]
    print(f"{'profile':<8} {'nodes':>6} {'epoch':>6} {'train s':>8} {'dir CE':>7} {'parent acc':>11}"
          f" {'query ms':>9} {'sim ms':>7}")
    for profile, row in sweep.items():
        print(f"{profile:<8} {row['nodes']:>6} {row['selected_epoch']:>6} {row['train_seconds']:>8.1f}"
              f" {row['direction_ce']:>7.3f} {row['parent_accuracy']:>11.3f}"
              f" {row['query_ms']:>9.3f} {row['simulate_ms']:>7.3f}")

    nodes = [row["nodes"] for row in sweep.values()]
    figure, axes = plt.subplots(1, 3, figsize=(14, 3.8))
    for key, axis, title in (
        ("collisions", axes[0], "Collision fraction"),
        ("dimension", axes[1], "Fractal dimension"),
        ("max_depth", axes[2], "Mean max depth"),
    ):
        for name in names:
            axis.plot(nodes, [row[key][name] for row in sweep.values()], "o-", label=name, lw=1.5)
        axis.set_xscale("log", base=2)
        axis.set_xticks(nodes, [str(n) for n in nodes])
        axis.set_xlabel("nodes")
        axis.set_title(title)
    axes[1].axhline(1.71, color="0.6", lw=0.8, ls="--")
    axes[0].legend(frameon=False)
    plt.tight_layout()
    plt.show()